In [17]:
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown

# Charger le fichier de données nettoyé
df_clean = pd.read_csv('../data_DVF_clean.csv')

# Ajouter la colonne Année pour l'analyse temporelle
df_clean['date_mutation'] = pd.to_datetime(df_clean['date_mutation'])
df_clean['annee'] = df_clean['date_mutation'].dt.year

# --- CORRECTION CLÉ : Standardiser le nom de la commune ---
df_clean['commune'] = df_clean['commune'].str.lower()
# --------------------------------------------------------

print(f"✅ Données chargées et prêtes pour l'analyse temporelle. Années disponibles : {df_clean['annee'].unique().tolist()}")

✅ Données chargées et prêtes pour l'analyse temporelle. Années disponibles : [2020, 2021, 2022, 2023, 2024, 2025]


In [18]:
def plot_evolution_prix(villes_selectionnees):
    """
    Calcule et affiche l'évolution du prix moyen au m² par an pour les villes sélectionnées.
    """
    if not villes_selectionnees:
        display(Markdown("⚠️ **Veuillez sélectionner au moins une ville pour visualiser l'évolution.**"))
        return
        
    # Calcul de l'évolution du prix moyen au m² par Année et par Commune
    df_evolution = df_clean[df_clean['commune'].isin(villes_selectionnees)]
    
    # Agrégation (Prix moyen au m² et nombre de transactions)
    df_agg = df_evolution.groupby(['annee', 'commune'])['prix_au_m2'].agg(['mean', 'count']).reset_index()
    
    # Nettoyage des données agrégées (filtrer les années/villes avec peu de transactions)
    df_agg = df_agg[df_agg['count'] >= 20] # Seuil de 20 transactions par an/ville pour la fiabilité
    
    # Visualisation (Matplotlib)
    plt.figure(figsize=(12, 6))
    
    # Tracer une ligne pour chaque ville sélectionnée
    for ville in villes_selectionnees:
        df_ville = df_agg[df_agg['commune'] == ville]
        if not df_ville.empty:
            plt.plot(df_ville['annee'], df_ville['mean'], marker='o', label=ville)

    plt.title('Évolution Annuelle du Prix Moyen au M² (Appartements T1-T3)', fontsize=16)
    plt.xlabel('Année')
    plt.ylabel('Prix Moyen au m² (€)')
    plt.grid(axis='y', linestyle='--')
    plt.legend(title='Commune')
    plt.xticks(df_agg['annee'].unique().astype(int))
    
    # Ajouter une note pour Hugo
    plt.figtext(0.5, 0.01, 
                "**Analyse pour Hugo :** Une courbe en forte croissance indique un bon potentiel de valorisation à long terme.", 
                ha="center", fontsize=10, bbox={"facecolor":"lightblue", "alpha":0.5, "pad":5})
    
    plt.show()

# Liste des villes éligibles 
villes_disponibles_exploration = sorted(df_clean['commune'].unique().tolist())

# --- Définition des valeurs par défaut SÛRES pour la Cellule 3 ---
# Utiliser les noms en minuscules pour la valeur par défaut
valeurs_par_defaut = [
    ville for ville in ['paris', 'lyon', 'marseille'] 
    if ville in villes_disponibles_exploration
]
# Si aucune des grandes villes n'est là (cas rare), utiliser les 3 premières.
if not valeurs_par_defaut and len(villes_disponibles_exploration) >= 3:
    valeurs_par_defaut = villes_disponibles_exploration[:3]
elif not valeurs_par_defaut and len(villes_disponibles_exploration) > 0:
    valeurs_par_defaut = [villes_disponibles_exploration[0]]

In [19]:
# Créer le widget (Sélection multiple)
villes_checklist = widgets.SelectMultiple(
    options=villes_disponibles_exploration,
    value=valeurs_par_defaut, # <-- UTILISATION DE LA VARIABLE SÛRE
    description='Villes à comparer :',
    disabled=False,
    layout=widgets.Layout(height='200px')
)

# Lier le widget à la fonction
interactive_evolution = widgets.interactive(
    plot_evolution_prix,
    villes_selectionnees=villes_checklist
)

# Afficher l'ensemble
display(Markdown('## 📈 Outil 3 : Tendance de l\'Évolution des Prix (Besoin n°3)'))
display(Markdown('**Instruction pour Hugo :** Sélectionnez des villes (Ctrl/Cmd + Clic) pour analyser leur historique de prix et identifier les zones à forte valorisation.'))
display(interactive_evolution)

## 📈 Outil 3 : Tendance de l'Évolution des Prix (Besoin n°3)

**Instruction pour Hugo :** Sélectionnez des villes (Ctrl/Cmd + Clic) pour analyser leur historique de prix et identifier les zones à forte valorisation.

interactive(children=(SelectMultiple(description='Villes à comparer :', index=(0, 1, 2), layout=Layout(height=…